# Atropos: Getting Started

This notebook demonstrates the basics of using Atropos to estimate ROI for LLM pruning and quantization.

## What you'll learn

- How to create custom deployment scenarios
- How to evaluate different optimization strategies
- How to interpret the results

## Setup

In [ ]:
# Install atropos if not already installed
# %pip install -e ..

from atropos import DeploymentScenario, OptimizationStrategy, estimate_outcome
from atropos.presets import SCENARIOS, STRATEGIES, QUANTIZATION_BONUS
from atropos.calculations import combine_strategies
import pandas as pd

## 1. Using Built-in Presets

Atropos comes with pre-configured scenarios and strategies.

In [ ]:
# View available scenarios
print("Available Scenarios:")
for name, scenario in SCENARIOS.items():
    print(f"  {name}: {scenario.parameters_b}B params, {scenario.memory_gb}GB RAM")

In [ ]:
# View available strategies
print("\nAvailable Strategies:")
for name, strategy in STRATEGIES.items():
    print(f"  {name}: {strategy.memory_reduction_fraction*100:.0f}% memory reduction, risk={strategy.quality_risk}")

In [ ]:
# Analyze a preset scenario with a preset strategy
scenario = SCENARIOS['medium-coder']
strategy = STRATEGIES['structured_pruning']

outcome = estimate_outcome(scenario, strategy)

print(f"Scenario: {outcome.scenario_name}")
print(f"Strategy: {outcome.strategy_name}")
print(f"\nMemory: {outcome.baseline_memory_gb:.1f}GB → {outcome.optimized_memory_gb:.1f}GB")
print(f"Throughput: {outcome.baseline_throughput_toks_per_sec:.0f} → {outcome.optimized_throughput_toks_per_sec:.0f} tok/s")
print(f"Annual Savings: ${outcome.annual_total_savings_usd:,.0f}")
print(f"Break-even: {outcome.break_even_years*12:.1f} months")

## 2. Creating Custom Scenarios

Define your own deployment scenario based on your actual infrastructure.

In [ ]:
# Define a custom deployment scenario
my_scenario = DeploymentScenario(
    name="my-production-api",
    parameters_b=70.0,                    # 70B parameter model
    memory_gb=48.0,                       # Current memory usage
    throughput_toks_per_sec=35.0,         # Current throughput
    power_watts=600.0,                    # Power draw
    requests_per_day=100000,              # Daily request volume
    tokens_per_request=1500,              # Avg tokens per request
    electricity_cost_per_kwh=0.12,        # $/kWh
    annual_hardware_cost_usd=50000.0,     # Hardware lease/cloud cost
    one_time_project_cost_usd=45000.0,    # Estimated optimization cost
)

print(f"Created scenario: {my_scenario.name}")
print(f"Daily tokens: {my_scenario.requests_per_day * my_scenario.tokens_per_request:,}")

In [ ]:
# Evaluate different strategies on custom scenario
results = []

for strategy_name, strategy in STRATEGIES.items():
    outcome = estimate_outcome(my_scenario, strategy)
    results.append({
        'Strategy': strategy_name,
        'Memory Reduction %': (1 - outcome.optimized_memory_gb/outcome.baseline_memory_gb) * 100,
        'Throughput Gain %': (outcome.optimized_throughput_toks_per_sec/outcome.baseline_throughput_toks_per_sec - 1) * 100,
        'Annual Savings $': outcome.annual_total_savings_usd,
        'Break-even (months)': outcome.break_even_years * 12 if outcome.break_even_years else None,
        'Quality Risk': outcome.quality_risk,
    })

df = pd.DataFrame(results)
df

## 3. Combining Strategies

Combine pruning with quantization for compound effects.

In [ ]:
# Compare pruning alone vs pruning + quantization
pruning_only = estimate_outcome(my_scenario, STRATEGIES['structured_pruning'])

pruning_plus_quant = estimate_outcome(
    my_scenario, 
    combine_strategies(STRATEGIES['structured_pruning'], QUANTIZATION_BONUS)
)

comparison = pd.DataFrame([
    {
        'Configuration': 'Baseline',
        'Memory (GB)': pruning_only.baseline_memory_gb,
        'Throughput (tok/s)': pruning_only.baseline_throughput_toks_per_sec,
        'Annual Cost ($)': pruning_only.baseline_annual_total_cost_usd,
    },
    {
        'Configuration': 'Structured Pruning',
        'Memory (GB)': pruning_only.optimized_memory_gb,
        'Throughput (tok/s)': pruning_only.optimized_throughput_toks_per_sec,
        'Annual Cost ($)': pruning_only.optimized_annual_total_cost_usd,
    },
    {
        'Configuration': 'Pruning + Quantization',
        'Memory (GB)': pruning_plus_quant.optimized_memory_gb,
        'Throughput (tok/s)': pruning_plus_quant.optimized_throughput_toks_per_sec,
        'Annual Cost ($)': pruning_plus_quant.optimized_annual_total_cost_usd,
    },
])

comparison['Savings vs Baseline'] = comparison['Annual Cost ($)'].iloc[0] - comparison['Annual Cost ($)']
comparison

## 4. Sensitivity Analysis

Understand how sensitive the ROI is to key assumptions.

In [ ]:
from atropos.core.calculator import ROICalculator

calc = ROICalculator()
calc.register_scenario(my_scenario)
calc.register_strategy(STRATEGIES['structured_pruning'])

# Vary memory_reduction_fraction from -20% to +20% of base value
sens_results = calc.sensitivity_analysis(
    'my-production-api', 
    'structured_pruning', 
    'memory_reduction_fraction',
    variations=4,
    step=0.05
)

sens_df = pd.DataFrame([
    {
        'Variation Factor': factor,
        'Memory Reduction %': outcome.optimized_memory_gb / outcome.baseline_memory_gb * 100,
        'Annual Savings ($)': outcome.annual_total_savings_usd,
        'Break-even (months)': outcome.break_even_years * 12 if outcome.break_even_years else None,
    }
    for factor, outcome in sens_results
])

sens_df

## 5. Exporting Results

Save results for reporting or further analysis.

In [ ]:
from atropos.io import render_report

# Generate markdown report
md_report = render_report(pruning_plus_quant, 'markdown')
print(md_report[:500] + "...")

# Save to file
with open('../reports/analysis_report.md', 'w') as f:
    f.write(md_report)

## Next Steps

- See `02_batch_analysis.ipynb` for analyzing multiple scenarios
- See `03_custom_strategies.ipynb` for defining your own optimization strategies